# DePlot — DIMER chart-to-table extraction tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/deplot-chart-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/deplot-chart-pipeline/blob/main/tutorials/deplot_chart_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google%2Fdeplot-ffcc4d?style=flat)](https://huggingface.co/google/deplot) [![Upstream](https://img.shields.io/badge/Upstream-google--research%2Fpix2struct-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/pix2struct) [![arXiv](https://img.shields.io/badge/arXiv-2212.10505-b31b1b.svg)](https://arxiv.org/abs/2212.10505)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** chart-to-table extraction — one chart image → its linearised data table (title, header row, data rows) — using the pinned `google/deplot` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/deplot_chart_pipeline/pipeline.py` at revision `7f0202dce276`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `6e76d62430da16986be3426bae32301fb9115397` (~1133 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the Pix2Struct image-encoder/text-decoder (a ViT-style encoder over variable-resolution 16×16 patches and a 12-layer text decoder, 282M parameters, pretrained by parsing masked web screenshots into HTML and fine-tuned by Google Research on plot-to-table data as DePlot) reads the fixed instruction **rendered as a text header above the chart** — the Pix2Struct convention — scales the composite to fill at most 2048 patches, and generates a linearised table: rows separated by the literal token `<0x0A>`, cells by `|`, with a leading `TITLE | …` row when a title was read. Decoding is greedy (`do_sample=False`) under a caller-owned `max_new_tokens` budget. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights, processor and tokenizer, and the carried module adds snapshot verification, the input contract (image side ceilings, the token budget), a fixed output contract, an offline header font (Pillow's bundled Aileron replaces the Hub font the upstream processor would otherwise download), `parse_table`, and the `cell_accuracy`, `validate_inputs` and `evaluation_report` helpers. The default sample is a bar chart drawn in code from a known data table, so relaxed cell accuracy and the title match are demonstration (plumbing) evidence for one chart, not a chart-to-table benchmark.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, draw a synthetic bar chart from a known table (or upload your own chart) and validate it into an input manifest, choose a token budget, run the supported task, read the linearised table correctly (separators, title row, parsed rows, the `truncated` flag, no score), exercise an optional BYOD path, produce an evaluation report that is `sample-sanity` with relaxed `cell_accuracy` and `title_match` only when the expected table exists and `not-measurable` otherwise, and export the table as JSON and CSV with provenance.

**This notebook does not demonstrate:** chart question answering or reasoning over the extracted table (the DePlot paper pairs the table with an LLM; none is bundled), charts in images with several plots or a plot embedded in a page (one chart image per call), any instruction other than the fixed plot-to-table prompt, batch throughput, sampling or beam search, evaluation on ChartQA/PlotQA or the paper's relative-mapping-similarity metric (only position-wise relaxed cell accuracy against a table you supply is computed here), and any training. The model generates a table for any image, including one with no chart, and gives no signal when it invents.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. CPU is adequate: the repository's model card records 5.4 s to load and 6.3 s for the 800×520 drawn bar chart (56 generated tokens) in the Windows venv (Intel Core Ultra 9 275HX); cost scales with the tokens generated. The pinned `torch==2.14.0` install and the 1.13 GB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python and PIL; what an encoder–decoder model's generated tokens are; how a chart's data table is laid out (header row, one row per category); that a well-formed table is not a correct one.
- **Data:** the default sample is a deterministic 800×520 bar chart drawn in code with Pillow's bundled font — a title, a labelled y axis with gridlines, four quarterly bars with x labels and value labels — from a five-row data table you can read in the code, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one image decodable by Pillow (PNG/JPEG/WebP and similar) of a **single chart** (bar, line or pie, ideally with axis labels and a title), any colour mode, sides between 16 and 4096 px. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `google/deplot` snapshot (~1133 MB in total) at revision `6e76d62430da…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'deplot-chart-pipeline',
    'repository_revision': '7f0202dce276919f0f3ba63304b6bcf0315b9f49',
    'embedded_module': 'src/deplot_chart_pipeline/pipeline.py',
    'embedded_modules': ['src/deplot_chart_pipeline/pipeline.py'],
    'module_sha256': 'bb507490ce02348384d5bb1f4134c299b6b72d0c2904f727d33274c1be6434a3',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/deplot_chart_pipeline/` @ `7f0202dce276`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/deplot_chart_pipeline/pipeline.py`

In [ ]:
"""Chart-to-table extraction with the pinned ``google/deplot`` checkpoint (DePlot, a Pix2Struct model).

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the Pix2Struct architecture comes from the pinned ``transformers`` release,
the weights are SafeTensors, and no model-repository code is executed. The fixed instruction is rendered
as a text header on top of the chart (the Pix2Struct VQA input convention) with Pillow's bundled font,
so no font is fetched from the Hub at inference time.
"""

from __future__ import annotations

import hashlib
import json
import re
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from PIL import Image, ImageFont

MODEL_ID = "google/deplot"
MODEL_REVISION = "6e76d62430da16986be3426bae32301fb9115397"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "deplot"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# The instruction the pinned README renders above the chart; DePlot was trained on this exact prompt
# and the pipeline does not expose any other.
INSTRUCTION = "Generate underlying data table of the figure below:"
# Generation ceilings. 512 is the max_new_tokens the pinned README's example passes; the ceiling
# leaves room for a long table.
MAX_NEW_TOKENS = 1024
DEFAULT_MAX_NEW_TOKENS = 512
DECODING = "greedy"
# Output conventions: DePlot linearises a table as rows separated by the literal token ``<0x0A>`` and
# cells separated by ``|``; the first row is ``TITLE | <chart title>`` when a title was read.
ROW_SEPARATOR = "<0x0A>"
CELL_SEPARATOR = "|"
# Input ceilings. The processor extracts at most MAX_PATCHES 16x16 patches (preprocessor_config.json)
# after scaling the image to fill that budget, so pixel count only guards memory during resizing.
MAX_PATCHES = 2048
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
# Relaxed numeric match tolerance for cell_accuracy (the ChartQA/DePlot "relaxed accuracy" convention).
RELATIVE_TOLERANCE = 0.05
_NUMBER_RE = re.compile(r"^[-+]?\$?\s*(\d[\d,]*\.?\d*|\.\d+)\s*%?$")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def header_font_bytes() -> bytes:
    """Pillow's bundled Aileron Regular (CC0) as TrueType bytes: the header font for the rendered question.

    The upstream image processor otherwise fetches ``ybelkada/fonts/Arial.TTF`` from the Hub at
    inference time — an unpinned, unlisted download of a proprietary font. The bundled subset covers
    the printable ASCII range, which is what a question is expected to use.
    """
    font = ImageFont.load_default(size=36)
    data = getattr(font, "font_bytes", None)
    if not data:
        raise RuntimeError("Pillow's bundled TrueType font is unavailable (FreeType support missing)")
    return bytes(data)


def parse_table(text: str) -> dict[str, Any]:
    """Split DePlot's linearised output into a title (or None) and a list of rows of stripped cells."""
    rows: list[list[str]] = []
    title: str | None = None
    for raw_row in text.split(ROW_SEPARATOR):
        cells = [cell.strip() for cell in raw_row.split(CELL_SEPARATOR)]
        if not any(cells):
            continue
        if title is None and not rows and len(cells) >= 2 and cells[0].upper() == "TITLE":
            title = CELL_SEPARATOR.join(cells[1:]).strip()
            continue
        rows.append(cells)
    return {
        "title": title,
        "rows": rows,
        "n_rows": len(rows),
        "n_columns": max((len(r) for r in rows), default=0),
    }


def _as_number(cell: str) -> float | None:
    match = _NUMBER_RE.match(cell.strip())
    if not match:
        return None
    try:
        return float(match.group(1).replace(",", ""))
    except ValueError:
        return None


def cells_match(predicted: str, expected: str, *, relative_tolerance: float = RELATIVE_TOLERANCE) -> bool:
    """Relaxed cell match: equal after case/whitespace normalisation, or numerically within the tolerance."""
    if " ".join(predicted.lower().split()) == " ".join(expected.lower().split()):
        return True
    p, e = _as_number(predicted), _as_number(expected)
    if p is None or e is None:
        return False
    return abs(p - e) <= relative_tolerance * abs(e) if e != 0 else abs(p) <= relative_tolerance


def cell_accuracy(
    predicted_rows: Sequence[Sequence[str]],
    expected_rows: Sequence[Sequence[str]],
    *,
    relative_tolerance: float = RELATIVE_TOLERANCE,
) -> dict[str, Any]:
    """Position-wise relaxed cell accuracy of a predicted table against the expected one.

    Every expected cell (row i, column j) counts once; it is matched only against the predicted cell at
    the same position (a missing row or column is a miss, an extra one is not penalised here but is
    reported through the shape fields). This is a sanity measure, not the paper's RMS metric.
    """
    if not expected_rows or not any(expected_rows):
        raise ValueError("expected_rows must contain at least one cell")
    total = matched = 0
    for i, expected in enumerate(expected_rows):
        predicted = predicted_rows[i] if i < len(predicted_rows) else []
        for j, cell in enumerate(expected):
            total += 1
            if j < len(predicted) and cells_match(predicted[j], cell, relative_tolerance=relative_tolerance):
                matched += 1
    return {
        "matched": matched,
        "total": total,
        "value": matched / total,
        "predicted_shape": [len(predicted_rows), max((len(r) for r in predicted_rows), default=0)],
        "expected_shape": [len(expected_rows), max((len(r) for r in expected_rows), default=0)],
        "relative_tolerance": relative_tolerance,
    }


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one chart image as PIL.Image.Image (any mode, converted to RGB): a bar, line or pie chart",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "instruction": INSTRUCTION,
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "decoding": f"{DECODING} (do_sample=False), deterministic on a fixed device and dtype",
    "preprocessing": (
        "the fixed instruction is rendered as a black-on-white header (Pillow's bundled font) above the "
        "chart; the composite is scaled to fill at most MAX_PATCHES 16x16 patches (aspect ratio preserved), "
        "normalised per image, and flattened into patch tokens with row/column positions; the decoder "
        "generates the linearised table"
    ),
    "output": (
        f"linearised table text (rows separated by {ROW_SEPARATOR!r}, cells by {CELL_SEPARATOR!r}, optional "
        "leading TITLE row) plus its parsed rows; no score"
    ),
}


def _check_inputs(image: Any, max_new_tokens: Any) -> tuple[Image.Image, int]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``extract_table`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f"max_new_tokens must be between 1 and MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
    return rgb, max_new_tokens


def validate_inputs(
    image: Image.Image,
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``extract_table`` would; a caller that wants the
    finding recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, checked_tokens = _check_inputs(image, max_new_tokens)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (extract_table takes one chart image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "instruction": INSTRUCTION,
        "generation": {"max_new_tokens": checked_tokens, "do_sample": False, "decoding": DECODING},
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    expected_rows: Sequence[Sequence[str]] | None = None,
    *,
    expected_title: str | None = None,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``expected_rows`` (the table the chart really encodes, header row first) the report carries
    position-wise relaxed ``cell_accuracy`` and, when ``expected_title`` is given, a ``title_match``
    entry, verdict ``sample-sanity``; without ``expected_rows`` it is ``not-measurable`` and says what
    labelled data would make the task measurable.
    """
    table = result.get("table") or parse_table(str(result["text"]))
    base = {
        "task": "chart image -> linearised data table (plot-to-table)",
        "score_semantics": (
            "the table is generated text and carries no score, probability or correctness signal; a "
            "well-formed table is not evidence that its numbers are read from the chart. Greedy decoding "
            "makes the output reproducible on a fixed device and dtype, a reproducibility property, not a "
            "quality one"
        ),
        "sample_kind": sample_kind,
        "predicted_shape": [table["n_rows"], table["n_columns"]],
        "truncated": result.get("truncated"),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if expected_rows is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no expected data table was supplied for the evaluated chart",
            "needs": (
                "chart images paired with their underlying data tables from the deployment domain (chart "
                "types, styles, renderers) scored with relaxed cell accuracy or the DePlot paper's relative "
                "mapping similarity; no such labelled set ships with this repository"
            ),
        }
    accuracy = cell_accuracy(table["rows"], expected_rows)
    metrics: list[dict[str, Any]] = [
        {
            "id": "cell_accuracy",
            "value": accuracy["value"],
            "matched": accuracy["matched"],
            "total": accuracy["total"],
            "predicted_shape": accuracy["predicted_shape"],
            "expected_shape": accuracy["expected_shape"],
            "normalisation": (
                "position-wise; text cells compared case/whitespace-insensitively, numeric cells within "
                f"{RELATIVE_TOLERANCE:.0%} relative tolerance"
            ),
            "estimation": "one chart, no dispersion estimate",
        }
    ]
    if expected_title is not None:
        metrics.append(
            {
                "id": "title_match",
                "value": 1.0 if table["title"] and cells_match(table["title"], expected_title) else 0.0,
                "predicted": table["title"],
                "expected": expected_title,
                "estimation": "one chart, structural sanity only",
            }
        )
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(metrics)} sanity measure(s) on one tutorial chart whose data you rendered yourself; "
            "plumbing evidence, not a chart-to-table benchmark"
        ),
        "needs": (
            "a labelled chart/table set from the deployment domain (chart types, styles, renderers, "
            "languages) for any plot-to-table accuracy claim"
        ),
    }


@dataclass
class DePlotPipeline:
    """``_runner(image, max_new_tokens)`` returns ``{"text": str, "new_tokens": int}``."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    dtype: str = "float32"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> DePlotPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        font_bytes = header_font_bytes()
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import Pix2StructForConditionalGeneration, Pix2StructProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = Pix2StructProcessor.from_pretrained(location, **common)
        if not getattr(processor.image_processor, "is_vqa", False):
            raise RuntimeError("snapshot image processor is not the VQA variant (is_vqa=False); refusing")
        model = Pix2StructForConditionalGeneration.from_pretrained(location, dtype=torch.float32, **common)
        model = model.eval().to(resolved_device)

        def runner(image: Image.Image, max_new_tokens: int) -> dict[str, Any]:
            # The image processor is called directly: Pix2StructProcessor.__call__ drops the
            # font_bytes kwarg, and font_bytes is what replaces the default Hub font download
            # (see header_font_bytes). The VQA processor renders the instruction as the header.
            inputs = processor.image_processor(
                image, header_text=INSTRUCTION, return_tensors="pt", font_bytes=font_bytes
            ).to(resolved_device)
            with torch.inference_mode():
                generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            decoded = processor.tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
            return {"text": decoded, "new_tokens": int(generated[0].shape[0]) - 1}

        return cls(runner, resolved_device, "float32", source)

    def extract_table(
        self,
        image: Image.Image,
        *,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    ) -> dict[str, Any]:
        """Translate one chart image into its linearised data table; ``table`` is the parsed form."""
        rgb, checked_tokens = _check_inputs(image, max_new_tokens)
        raw = self._runner(rgb, checked_tokens)
        if not isinstance(raw, dict) or "text" not in raw:
            raise RuntimeError("runner must return a dict with 'text'")
        text = str(raw["text"]).strip()
        new_tokens = int(raw.get("new_tokens", 0))
        return {
            "text": text,
            "table": parse_table(text),
            "instruction": INSTRUCTION,
            "image_size": list(rgb.size),
            "new_tokens": new_tokens,
            "truncated": new_tokens >= checked_tokens,
            "generation": {"max_new_tokens": checked_tokens, "do_sample": False, "decoding": DECODING},
            "device": self.device,
            "dtype": self.dtype,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `6e76d62430da…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `DePlotPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "deplot",
  "modelId": "google/deplot",
  "revision": "6e76d62430da16986be3426bae32301fb9115397",
  "files": [
    {
      "path": "README.md",
      "bytes": 4317,
      "sha256": "419a84efe72647da6e56637a4b92394b3b3df6313d8b233c5ddc9942aad80102"
    },
    {
      "path": "config.json",
      "bytes": 4883,
      "sha256": "f1dc8bcbda2ac4f8715c2d5003d3afb591ac0af97782fdf95a65cf26f805c5bb"
    },
    {
      "path": "model.safetensors",
      "bytes": 1129177976,
      "sha256": "ab90055611f42fee327d9ecf3c9cdac63e847bd19a0ac8ea86b0e8134fe0711b"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 249,
      "sha256": "c84e4eebc84171d6069533d9f0147ec7b4afd02ab78697cb5c30f9419ef7dc45"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 2201,
      "sha256": "5c87151ef0f72a99d1f766a4c418bd2a1f90aaa30a8e22fe5eca9641daebb64f"
    },
    {
      "path": "spiece.model",
      "bytes": 851388,
      "sha256": "7fd650335add59bed55a432186ca0437a09e185c2d241faab468a538fe6bcf94"
    },
    {
      "path": "tokenizer.json",
      "bytes": 3265159,
      "sha256": "0af109b23840545ef2c286073f4373959badba1faa73c8557881d5126f6287c9"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 2623,
      "sha256": "73949c3b853a39cd4661303b0e5cb7b836f7d783be6e898f8b7ae89e203cbe73"
    }
  ],
  "totalBytes": 1133308796
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = DePlotPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Draw the synthetic bar chart or optional BYOD

The default sample is **synthetic** and carries its own reference: a bar chart — the title `Quarterly revenue 2025 (USD millions)`, a y axis from 0 to 200 with gridlines and tick labels, four bars for Q1–Q4 with their values printed above them and an x-axis caption — is drawn with Pillow's bundled font at 800×520 from the five-row table `EXPECTED_TABLE` (header row first), the same chart the repository's smoke run used. That table and the title are the references for the `cell_accuracy` and `title_match` sanity checks later; they are not a labelled dataset, so nothing here is a benchmark, and a drawn chart with printed values is far easier than a typical published chart. The image digest is printed for the record. BYOD is optional and disabled by default; when enabled, upload one chart image — no expected table exists for it, so the evaluation report will be `not-measurable`.

The token budget is a **caller-owned request parameter**: `max_new_tokens` bounds the linearised table (`DEFAULT_MAX_NEW_TOKENS = 512`, the pinned README example's value; `MAX_NEW_TOKENS = 1024` is the ceiling). The instruction is fixed — DePlot was trained on exactly `INSTRUCTION` and the pipeline exposes no other. Nothing is validated in this cell — the next section hands the image and the budget to the pipeline's own validation stage, which is the only checker. Look for a dictionary naming the sample kind, the image size and digest, the budget and the expected table's shape.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
max_new_tokens = 512  # @param {type:"integer"}

CHART_TITLE = 'Quarterly revenue 2025 (USD millions)'
EXPECTED_TABLE = [['Quarter', 'Revenue'], ['Q1', '120'], ['Q2', '135'], ['Q3', '150'], ['Q4', '180']]


def bar_chart(width=800, height=520):
    """Titled bar chart with a labelled y axis, gridlines, x labels and value labels, drawn from EXPECTED_TABLE."""
    img = Image.new('RGB', (width, height), 'white')
    d = ImageDraw.Draw(img)
    title_f, tick_f, label_f = ImageFont.load_default(size=24), ImageFont.load_default(size=16), ImageFont.load_default(size=18)
    d.text((width / 2 - d.textlength(CHART_TITLE, font=title_f) / 2, 24), CHART_TITLE, fill='black', font=title_f)
    x0, y0, x1, y1 = 100, 80, width - 40, height - 80
    d.line([(x0, y0), (x0, y1), (x1, y1)], fill='black', width=2)
    ymax = 200
    for v in range(0, ymax + 1, 50):
        y = y1 - (y1 - y0) * v / ymax
        d.line([(x0, y), (x1, y)], fill=(200, 200, 200), width=1)
        d.text((x0 - 12 - d.textlength(str(v), font=tick_f), y - 9), str(v), fill='black', font=tick_f)
    n = len(EXPECTED_TABLE) - 1
    slot = (x1 - x0) / n
    for i, (label, value) in enumerate(EXPECTED_TABLE[1:]):
        bx0 = x0 + slot * i + slot * 0.25
        bx1 = bx0 + slot * 0.5
        by = y1 - (y1 - y0) * int(value) / ymax
        d.rectangle([bx0, by, bx1, y1], fill=(60, 110, 200), outline=(20, 50, 120))
        d.text(((bx0 + bx1) / 2 - d.textlength(label, font=label_f) / 2, y1 + 12), label, fill='black', font=label_f)
        d.text(((bx0 + bx1) / 2 - d.textlength(value, font=tick_f) / 2, by - 22), value, fill='black', font=tick_f)
    d.text((width / 2 - 40, height - 40), 'Quarter', fill='black', font=label_f)
    return img


if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    expected_table, expected_title = None, None
    sample_kind = 'BYOD'
else:
    # Deterministic synthetic chart: no randomness, so no seed is needed and the digest is stable per Pillow build.
    image = bar_chart()
    expected_table, expected_title = EXPECTED_TABLE, CHART_TITLE
    image_name = 'synthetic_bar_chart_800x520.png'
    sample_kind = 'synthetic'

image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'mode': image.mode, 'size': image.size, 'rgb_sha256': image_sha256, 'max_new_tokens': max_new_tokens, 'expected_shape': None if expected_table is None else [len(expected_table), len(expected_table[0])]})

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `extract_table` applies — image type and sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px and `max_new_tokens` in `[1, MAX_NEW_TOKENS]` — and returns an **input manifest** naming the schema (including the fixed instruction, the header-rendering preprocessing, the output conventions and the decoding rule), the input's observed mode and size, the budget and the verdict. The manifest is written to `outputs/deplot_chart_input_manifest.json`. To show what rejection looks like, the cell also validates a zero budget and records the pipeline's own error message as a finding. Inside the pipeline the image is converted to RGB, the instruction is rendered above it, and the composite is scaled to the patch budget; nothing else is dropped or altered. The pipeline cannot tell whether the image is a chart: that contract is the caller's.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_PATCHES': MAX_PATCHES, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'DECODING': DECODING, 'INSTRUCTION': INSTRUCTION, 'ROW_SEPARATOR': ROW_SEPARATOR, 'CELL_SEPARATOR': CELL_SEPARATOR}})
input_manifest = validate_inputs(image, max_new_tokens=max_new_tokens, names=[image_name])
# Demonstrate rejection on a request that breaks the contract; the finding is recorded, not swallowed.
try:
    validate_inputs(image, max_new_tokens=0)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'zero-budget-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/deplot_chart_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Extract the table and read the output correctly

`extract_table` returns a dict with `text` — the linearised table exactly as decoded — `table` (the parsed form from `parse_table`: `title` or `None`, `rows` of stripped cells, `n_rows`, `n_columns`), the fixed `instruction`, `image_size`, `new_tokens`, a `truncated` flag that is true when the budget was exhausted, the generation settings and the model identity. **No score exists**: the table is generated text with no probability and no correctness signal, and a well-formed table is not evidence that its numbers were read from the chart. Greedy decoding is deterministic on a fixed device and dtype; CUDA kernel selection can change a token and therefore the rest of the table, so GPU and CPU outputs need not match. As recorded in the model card, the repository's CPU smoke on this same chart generated 56 tokens in 6.3 s: the title exactly, all four quarters and values exactly, and the header `Quarter | Quarterly revenue` — the y axis has no label, so the model filled the second header cell from the title, the one cell the reference calls `Revenue`. That is one observation on a drawn chart with printed values, not a calibration point.

In [ ]:
import time

t0 = time.time()
result = pipe.extract_table(image, max_new_tokens=max_new_tokens)
elapsed = time.time() - t0
table = result['table']
print({'seconds': round(elapsed, 1), 'new_tokens': result['new_tokens'], 'truncated': result['truncated'], 'device': pipe.device, 'title': table['title'], 'shape': [table['n_rows'], table['n_columns']]})
print(result['text'])
for row in table['rows']:
    print(' | '.join(f'{cell:>14}' for cell in row))
if result['truncated']:
    print('The token budget was exhausted: the table is incomplete. Raise max_new_tokens (ceiling MAX_NEW_TOKENS) and rerun.')

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. No chart-to-table metric is reported by default: the DePlot paper's relative mapping similarity and ChartQA-style accuracy need chart images paired with their data tables, and this repository ships none. The repository's metric helper is `cell_accuracy` — position-wise relaxed matching of every expected cell against the predicted cell at the same position, text cells compared case/whitespace-insensitively and numeric cells within 5 % relative tolerance (the ChartQA relaxed-accuracy convention), reported with both tables' shapes — plus a `title_match` entry when an expected title is supplied; with an expected table the verdict is `sample-sanity`. On the synthetic path that table is data **you rendered yourself** with the values printed on the bars, so a high accuracy proves only that the input contract, header rendering, forward pass, decoding and parsing round-trip. On BYOD no expected table exists, the verdict is `not-measurable`, and the report states what would make the task measurable. The report is written to `outputs/deplot_chart_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, expected_table, expected_title=expected_title, sample_kind=sample_kind)
with open('outputs/deplot_chart_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in report.items() if k != 'metrics'}, indent=2))
for metric in report['metrics']:
    if metric['id'] == 'cell_accuracy':
        print(f"cell_accuracy {metric['value']:.3f}  ({metric['matched']}/{metric['total']} cells; predicted shape {metric['predicted_shape']}, expected {metric['expected_shape']})")
    else:
        print(f"title_match   {metric['value']:.0f}  (predicted {metric['predicted']!r}, expected {metric['expected']!r})")
if report['verdict'] == 'not-measurable':
    print('No expected table exists for this input, so nothing is scored; compare the rows with the chart yourself.')

## 8. Export outputs and provenance

Machine-readable JSON preserves the full result (linearised text, parsed table, `new_tokens`, `truncated`, the budget), the evaluation report, the input manifest, the sample identity, digest, expected table and title, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, device). The parsed table is also written as CSV (one line per row, the title as a leading comment line when present), and a side-by-side PNG places the chart above a panel with the extracted rows for visual inspection — a supplement to, not a replacement for, the machine-readable files. No credentials are recorded.

In [ ]:
import csv

panel_lines = [' | '.join(row) for row in table['rows']][:16] or ['(no rows)']
panel_height = 24 + 22 * len(panel_lines)
annotated = Image.new('RGB', (image.width, image.height + panel_height), 'white')
annotated.paste(image.convert('RGB'), (0, 0))
draw = ImageDraw.Draw(annotated)
draw.line([(0, image.height + 1), (image.width, image.height + 1)], fill=(120, 120, 120), width=2)
panel_font = ImageFont.load_default(size=16)
for index, line in enumerate(panel_lines):
    draw.text((16, image.height + 10 + 22 * index), line[:120], fill=(40, 90, 220), font=panel_font)
annotated.save('outputs/deplot_chart_annotated.png')
with open('outputs/deplot_chart_table.csv', 'w', encoding='utf-8', newline='') as handle:
    if table['title']:
        handle.write(f"# title: {table['title']}\n")
    csv.writer(handle).writerows(table['rows'])
payload = {
    'prediction': result,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'expected_table': expected_table, 'expected_title': expected_title},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/deplot_chart_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The table is what the model generates after reading the chart with the instruction printed above it; nothing in the output scores it, and a tidy table can carry misread values, invented headers or missing rows. On the synthetic chart the `cell_accuracy` and `title_match` values in the evaluation report compare the output with a table you rendered yourself — with every value printed on its bar — and the verdict is `sample-sanity`, which proves only that the input contract, header rendering, forward pass, decoding and parsing work (the repository's smoke run matched 9 of 10 cells, the miss being the unlabelled y-axis header); they say nothing about charts without value labels, stacked or grouped bars, line charts with many points, pie charts, log axes, legends, unusual styles or non-English labels, and a BYOD result is a single-chart observation with the verdict `not-measurable`. **The model generates a table for any image**: a blank image yielded `TITLE | <0x0A> | % <0x0A> Eli | 55.1 <0x0A> Sarah | 44.9` in the smoke run — an invented two-row table with no signal — so an image that is not a chart produces a confident fabrication rather than an empty result, and `truncated` is the only structural flag you get. The pipeline provides no chart QA, no reasoning, no multi-chart pages, no alternative instructions and no training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated request, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** remove the value labels from `bar_chart` (delete the `d.text(... value ...)` line) and see how many cells survive when the model must read bar heights against the axis; add a y-axis label `Revenue` and check whether the header cell is fixed; lower `max_new_tokens` to 20 and watch `truncated` turn true; enable `USE_BYOD` with a published chart whose data you know, then pass the table as `expected_table` to `evaluation_report` to see the verdict switch to `sample-sanity`.

## References

- Repository README: https://github.com/kurtvalcorza/deplot-chart-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/deplot-chart-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/deplot-chart-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/google/deplot
- Upstream code: https://github.com/google-research/pix2struct
- DePlot: One-shot visual language reasoning by plot-to-table translation (Liu et al., 2022): https://arxiv.org/abs/2212.10505
- Pix2Struct: Screenshot Parsing as Pretraining for Visual Language Understanding (Lee et al., 2022): https://arxiv.org/abs/2210.03347
- ChartQA: A Benchmark for Question Answering about Charts with Visual and Logical Reasoning (Masry et al., 2022): https://arxiv.org/abs/2203.10244